<a href="https://colab.research.google.com/github/aathifsk1-gh/flyrank-assignment/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aathifsk1-gh/flyrank-assignment01/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
# --- Setup ---
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv")
print("Ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Ready.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The playbook hands an editor a ranked queue: each row is a page, a suggested action, and short reason codes in plain language.

Score blends the transparent baseline (visibility, freshness, position, depth) with a model probability of decline so the top of the list is both explainable and sharper than a pure rule.

Code (build queue):

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv").copy()
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

for c in ["word_count", "avg_position", "ctr", "engagement_rate", "scroll_rate",
          "sessions_90d", "search_volume", "competition", "cpc"]:
    df[c] = pd.to_numeric(df.get(c, 0), errors="coerce").fillna(0)

def percentile_rank(s):
    return s.rank(pct=True, method="average").fillna(0)

def normalize(s):
    s = s.astype(float)
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi != lo else pd.Series(0.0, index=s.index)

# Baseline sub-scores
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"] * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"])) * df["visibility_score"]
df["baseline_score"] = (
    0.40 * df["visibility_score"] + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"] + 0.05 * df["depth_gap_score"]
).clip(0, 1)

# Reason codes
def reason_codes(row):
    r = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        r.append("stale_visible_page")
    if str(row["trend_direction"]).lower() == "down" and row["impressions_90d"] >= 100:
        r.append("declining_with_demand")
    if 0 < row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        r.append("thin_visible_page")
    if 0 < row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        r.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        r.append("low_ctr_visible_page")
    if not r:
        r.append("general_refresh_review")
    return "|".join(r)

def suggested_action(reasons, model_p):
    rs = set(reasons.split("|"))
    if "thin_visible_page" in rs:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in rs:
        return "refresh_and_review_ctr"
    if model_p >= 0.65 or "declining_with_demand" in rs or "stale_visible_page" in rs:
        return "refresh"
    if model_p < 0.35:
        return "monitor"
    return "review"

df["reason_codes"] = df.apply(reason_codes, axis=1)

# Model score (client-holdout trained, applied to all for the playbook queue)
num = ["impressions_90d", "clicks_90d", "sessions_90d", "ctr", "avg_position",
       "engagement_rate", "word_count", "content_age_days", "days_since_last_update"]
cat = ["content_type", "position_tier"]
for c in cat:
    df[c] = df.get(c, "unknown").fillna("unknown").astype(str)
X = pd.concat([df[num], pd.get_dummies(df[cat], prefix=cat, dtype=float)], axis=1)
y = df["is_declining_label"]
clients = df["client_id"]
tr_c, _ = train_test_split(clients.unique(), test_size=0.2, random_state=RANDOM_STATE)
rf = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=5,
                            random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X.loc[clients.isin(tr_c)], y.loc[clients.isin(tr_c)])
df["model_decline_proba"] = rf.predict_proba(X)[:, 1]

# Blend for final priority (explainable + learned)
df["priority_score"] = (0.45 * df["baseline_score"] + 0.55 * df["model_decline_proba"]).clip(0, 1)
df["suggested_action"] = df.apply(lambda r: suggested_action(r["reason_codes"], r["model_decline_proba"]), axis=1)
df["queue_rank"] = df["priority_score"].rank(method="first", ascending=False).astype(int)

queue = df.sort_values("queue_rank")
print("Queue size:", len(queue))
print(queue[["queue_rank", "content_id", "priority_score", "suggested_action", "reason_codes",
             "impressions_90d", "is_declining_label"]].head(15).to_string(index=False))
print("\nTop-50 declining rate:", round(queue.head(50)["is_declining_label"].mean(), 3))
print("Action mix (top 100):")
print(queue.head(100)["suggested_action"].value_counts())

Queue size: 30000
 queue_rank           content_id  priority_score       suggested_action                                                   reason_codes  impressions_90d  is_declining_label
          1 content_f9d82e71e363        0.872641 refresh_and_review_ctr                     declining_with_demand|low_ctr_visible_page            24260                   1
          2 content_dcb529347579        0.867452 refresh_and_review_ctr declining_with_demand|page_one_decay_risk|low_ctr_visible_page            12512                   1
          3 content_e828799cd882        0.867152 refresh_and_review_ctr                     declining_with_demand|low_ctr_visible_page            13813                   1
          4 content_87f1ffe0bedb        0.854708 refresh_and_review_ctr                     declining_with_demand|low_ctr_visible_page             8225                   1
          5 content_8c6cdc6b2e1e        0.852142 refresh_and_review_ctr                     declining_with_demand|low_ctr_

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Who uses this: content editors / SEO leads planning the weekly refresh list.

What they do: open the ranked queue, read reason codes, pick the next pages to update, expand, or CTR-tune.Where it stops being valid:

One anonymized snapshot — not live site data and not every client’s full history.

Label is same-window trend, not a true future outcome.
Does not know business constraints (legal pages, seasonal campaigns, intentional thin pages).

Does not prove that acting causes recovery — it only prioritises review.
Scores will drift as Google, seasonality, and the portfolio change.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting, a person should check:**
Is this page still strategically important (product, revenue, brand)?

Was the “decline” expected (campaign ended, seasonality)?

Is there already a refresh in progress?

Do reason codes match what you see on the live page?

**No-go list (never fully automate):**

Publishing or overwriting live content without human review

Deleting or merging pages based only on the score

Changing titles/meta at scale without a sample QA

Using hashed IDs or this export to identify real clients

Claiming the score “knows Google’s algorithm”

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Treat the playbook as stale when any of these show up:**

Precision@50 on a fresh labeled slice falls close to the base rate or below the frozen baseline.

Action mix shifts hard (e.g. almost everything becomes monitor or almost everything refresh) without a portfolio change.

Feature distributions drift (median impressions, age, CTR) far from the training snapshot.

Editors override the top of the queue most of the time — the ranking no longer matches judgment.

New content types or tracking appear that the feature set never saw.


**Practical cadence: re-score weekly; full retrain + validation audit when a trigger fires or at least monthly on a new snapshot.**

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

os.makedirs("work/outputs", exist_ok=True)

export_cols = [
    "queue_rank", "content_id", "client_id", "priority_score", "baseline_score",
    "model_decline_proba", "suggested_action", "reason_codes", "is_declining_label",
    "impressions_90d", "avg_position", "ctr", "content_age_days",
    "days_since_last_update", "word_count", "trend_direction", "content_type"
]
path_queue = "work/outputs/action_playbook_queue.csv"
queue[export_cols].to_csv(path_queue, index=False)
print("Wrote", path_queue, "rows:", len(queue))

# Small summary table for the paper
summary = pd.DataFrame({
    "metric": ["rows", "top50_declining_rate", "top100_declining_rate", "base_rate"],
    "value": [
        len(queue),
        round(queue.head(50)["is_declining_label"].mean(), 3),
        round(queue.head(100)["is_declining_label"].mean(), 3),
        round(queue["is_declining_label"].mean(), 3),
    ]
})
path_sum = "work/outputs/action_playbook_summary.csv"
summary.to_csv(path_sum, index=False)
print("Wrote", path_sum)
display(summary)
display(queue.head(10)[export_cols[:8]])


Wrote work/outputs/action_playbook_queue.csv rows: 30000
Wrote work/outputs/action_playbook_summary.csv


,metric,value
0,rows,30000.000
1,top50_declining_rate,1.000
2,top100_declining_rate,0.980
3,base_rate,0.542


,queue_rank,content_id,client_id,priority_score,baseline_score,model_decline_proba,suggested_action,reason_codes
22592,1,content_f9d82e71e363,client_19581e27de,0.872641,0.908853,0.843013,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page
3710,2,content_dcb529347579,client_19581e27de,0.867452,0.874844,0.861405,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...
28731,3,content_e828799cd882,client_19581e27de,0.867152,0.856865,0.875569,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page
12269,4,content_87f1ffe0bedb,client_19581e27de,0.854708,0.823030,0.880626,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page
5046,5,content_8c6cdc6b2e1e,client_19581e27de,0.852142,0.866844,0.840113,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page
11788,6,content_8f65a4dbfd0e,client_19581e27de,0.849859,0.861686,0.840182,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...
28371,7,content_6e098546a7a2,client_19581e27de,0.846248,0.841448,0.850176,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page
23718,8,content_a0eeb9201f04,client_19581e27de,0.844971,0.803417,0.878969,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page
12930,9,content_6aa43079fb0c,client_3fdba35f04,0.840444,0.825477,0.852690,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page
21168,10,content_79bb85959143,client_19581e27de,0.833227,0.835172,0.831636,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page


Note: CSVs under work/ are often gitignored. For the paper, copy key numbers into markdown tables; keep only small summary metrics committed if CI allows.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.